In [1]:
import warnings
from pathlib import Path

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

In [3]:
from incidencia_delictiva.config import PROJ_ROOT, DATA_DIR, DATABASE_PATH

In [4]:
# configs
warnings.filterwarnings('ignore')

In [5]:
# scikit learn imports

# Dataset para el Modelado

Actualmente se cuenta con la base de datos `incidencia_delictiva`, la cual contiene 4 tablas:

* `delitos`
* `marginacion`
* `pobreza`
* `geodata`

En [`docs/data/`](../docs/data/) se encuentra la descripción detallada de cada uno de los datasets.

En esta notebook se propone un nuevo dataset, similar al conjunto desarrollado en la libreta [`data_integration.ipynb`](data_integration.ipynb). A diferencia de la propuesta anterior, ahora se incluirá la información geográfica y se seleccionarán variables que respondan de mejor manera a la pregunta conductora del proyecto:

> ¿Qué variables socioeconómicas explican y permiten predecir la incidencia delictiva a nivel municipal en México?

Además, se tomarán en cuenta los resultados obtenidos en la libreta [`clustering_socioeconomico.ipynb`](clustering_socioeconomico.ipynb), donde se observó una **alta correlación entre múltiples variables socioeconómicas**, lo que sugiere la presencia de redundancia en los datos.


## Variable objetivo

Primero se define la variable objetivo:

$$
\text{tasa\_delitos} = \frac{\text{total\_anual}}{\text{poblacion\_total}}\times 100000
$$

Esta variable se construye a partir de:

* `delitos.total_anual` (agregado por municipio-año)
* `marginacion.poblacion_total`

Se elige esta definición porque **controla por el tamaño poblacional**, permitiendo comparar municipios grandes y pequeños en la misma escala.


## Selección de variables

A continuación se describe la selección de variables por cada tabla.

### `delitos` (agregado municipio-año)

Variables seleccionadas:

* `total_anual` (para construir el target)
* `año`

De estas variables se deriva:

* `tasa_delitos` (variable objetivo)

No se incluyen desagregaciones por tipo de delito en esta primera aproximación para evitar alta dimensionalidad.

### `geodata`

Este dataset proporciona el **contexto espacial y estructural**. En la notebook [`datos_geograficos.ipynb`](datos_geograficos.ipynb) se describen sus detalles.

Variables seleccionadas:

* `area_km2`

  Representa el tamaño territorial del municipio.


  La intuición sugiere que municipios más grandes &rarr; menor densidad &rarr; patrones distintos de criminalidad.

* `region`

  Clasificación por [regiones de México](https://es.wikipedia.org/wiki/Regiones_de_M%C3%A9xico).


  Permite capturar efectos estructurales amplios (económicos, culturales y geográficos).

* `zona_metropolitana (0/1)`

  Indica si el municipio pertenece a una zona metropolitana (según CONAPO).


  Funciona como conector de urbanización e integración económica.

* `es_frontera (0/1)`


  Señala si el municipio pertenece a una entidad fronteriza.

  Captura dinámicas como movilidad, comercio, crimen organizado,etc.

* No se incluye `geometry` en esta etapa, aunque se utilizará posteriormente para construir variables espaciales más avanzadas.

### `marginacion`

Este dataset contiene variables que miden **vulnerabilidad social**.

En el análisis exploratorio se observó que el índice de marginación presenta **alta correlación con las demás variables**, lo que indica que resume gran parte de la información disponible.

Variables seleccionadas:

* `indice_marginacion_normalizado_2020`

  Indicador sintético de marginación.

* `porcentaje_analfabetismo`

  Proxy de capital humano.

* `porcentaje_sin_agua_entubada`

  Proxy de infraestructura básica.

* `porcentaje_viviendas_hacinamiento`

  Proxy de condiciones de vivienda.

Esta selección busca un balance entre:

* mantener un indicador global (índice)
* incluir variables específicas interpretables

Lo anterior introduce cierta colinealidad de forma intencional, útil en esta etapa exploratoria.

### `pobreza`

Este dataset contiene un gran número de variables altamente correlacionadas entre sí y con el índice de marginación.

Dado que el objetivo es mantener interpretabilidad sin introducir redundancia excesiva, se seleccionan variables que capturan **diferentes dimensiones de la pobreza**:

* `fn_pobreza_porcentaje`

  Nivel general de pobreza.

* `fn_pobreza_extrema_porcentaje`

  Condición de pobreza severa.

* `fn_vulnerable_ingresos_porcentaje`

  Población vulnerable por ingresos (riesgo de caer en pobreza).

* `fn_carencia_seguridad_social_porcentaje`

   Proxy de informalidad laboral y acceso a protección social.

* `poblacion_urbano`

  Se utilizará para construir una variable derivada:

  $$
  \text{prop\_urbano} = \frac{\text{poblacion\_urbano}}{\text{poblacion\_total}}
  $$

Esta variable permite capturar el grado de urbanización del municipio.

## Comentarios finales sobre la selección

* Se permite cierta **redundancia controlada** entre variables (especialmente entre marginación y pobreza) para evaluar su impacto en modelos posteriores.
* Se priorizan variables:

  * interpretables
  * agregadas a nivel municipal
  * disponibles de forma consistente

Este dataset constituye un **baseline**, sobre el cual se construirán versiones más complejas incorporando características espaciales y transformaciones adicionales.


## Construcción del dataset

A continuación se construye el dataset descrito anteriormente. 

El pipeline para el joining de los datos es el siguiente: 

1. delitos (agregado)
2. join marginacion (población + variables)
3. join pobreza (variables socioeconómicas)
4. join geodata (contexto espacial)
5. feature engineering (tasa + proporciones)

Definimis el query: 

In [6]:
query_dataset = """
WITH delitos_agregados AS (
    SELECT
        año,
        cvegeo,
        SUM(total_anual) AS total_delitos
    FROM delitos
    GROUP BY año, cvegeo
),

base AS (
    SELECT
        d.año,
        d.cvegeo,

        -- target base
        d.total_delitos,

        -- MARGINACIÓN
        m.poblacion_total,
        m.indice_marginacion_normalizado_2020,
        m.porcentaje_analfabetismo,
        m.porcentaje_sin_agua_entubada,
        m.porcentaje_viviendas_hacinamiento,

        -- POBREZA
        p.fn_pobreza_porcentaje,
        p.fn_pobreza_extrema_porcentaje,
        p.fn_vulnerable_ingresos_porcentaje,
        p.fn_carencia_seguridad_social_porcentaje,
        p.fn_poblacion,
        p.poblacion_urbano,

        -- GEODATA
        g.nomgeo,
        g.area_km2,
        g.region,
        g.zona_metropolitana,
        g.es_frontera

    FROM delitos_agregados d

    LEFT JOIN marginacion m
        ON d.cvegeo = m.cvegeo

    LEFT JOIN pobreza p
        ON d.cvegeo = p.cvegeo

    LEFT JOIN geodata g
        ON d.cvegeo = g.cvegeo
),

features AS (
    SELECT
        *,
        
        -- TARGET
        (total_delitos * 100000.0) / NULLIF(poblacion_total, 0) AS tasa_delitos,

        -- FEATURE ENGINEERING
        (poblacion_urbano * 1.0) / NULLIF(fn_poblacion, 0) AS prop_urbano

    FROM base
)

SELECT *
FROM features
WHERE poblacion_total IS NOT NULL;
"""

Establecemos conexión a la db: 

In [7]:
# database
con = duckdb.connect(DATABASE_PATH)

Probamos conexión a la db: 

In [8]:
result = con.execute("SELECT 42 AS test_connection").fetchone()

assert result == (42,), 'Fallo la conexión!'

In [9]:
dataset_modelado = con.execute(query_dataset).fetchdf()

In [10]:
dataset_modelado.info()

<class 'pandas.DataFrame'>
RangeIndex: 26038 entries, 0 to 26037
Data columns (total 21 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   año                                      26038 non-null  int64  
 1   cvegeo                                   26038 non-null  str    
 2   total_delitos                            26038 non-null  float64
 3   poblacion_total                          26038 non-null  int64  
 4   indice_marginacion_normalizado_2020      26038 non-null  float64
 5   porcentaje_analfabetismo                 26038 non-null  float64
 6   porcentaje_sin_agua_entubada             26038 non-null  float64
 7   porcentaje_viviendas_hacinamiento        26038 non-null  float64
 8   fn_pobreza_porcentaje                    26017 non-null  float64
 9   fn_pobreza_extrema_porcentaje            26017 non-null  float64
 10  fn_vulnerable_ingresos_porcentaje        26017 non-null  

## Conclusión

Se construyó un **dataset base de modelado (baseline)** a nivel municipio-año, integrando información de incidencia delictiva, condiciones socioeconómicas y contexto geográfico. El conjunto final contiene 26,038 observaciones y un total de 21 variables, incluyendo la variable objetivo `tasa_delitos`.

Este dataset permite capturar distintas dimensiones relevantes del fenómeno:

* **Incidencia delictiva:** mediante la tasa de delitos normalizada por población
* **Condiciones socioeconómicas:** a través de variables de marginación y pobreza
* **Estructura territorial:** incorporando área, región, pertenencia a zona metropolitana y condición de frontera
* **Urbanización:** mediante la proporción de población urbana (`prop_urbano`)

En conjunto, este baseline proporciona una primera aproximación estructurada para analizar la relación entre variables socioeconómicas y criminalidad a nivel municipal.
